# ASR Multi-Model Benchmark (Kaggle)

Automatically benchmarks **11 ASR models** (PhoWhisper + Whisper) on Vietnamese speech datasets.

## Setup
1. **Enable GPU**: Settings > Accelerator > **GPU T4 x2**
2. **Add HuggingFace secret** (optional, for pushing results): Settings > Secrets > Add `HF_TOKEN`
3. **Run all cells**
4. **Schedule** (optional): File > Schedule > Weekly

Results are saved as Kaggle output and optionally pushed to HuggingFace.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets[audio] jiwer accelerate soundfile librosa huggingface_hub tqdm pyyaml

## 2. Check GPU

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: Enable GPU in Settings > Accelerator")

## 3. Configuration

Edit these settings to customize the benchmark. Add/remove models and datasets as needed.

In [ ]:
# === MODELS TO BENCHMARK ===
MODELS = [
    # Vietnamese-specific (VinAI PhoWhisper)
    {"id": "vinai/PhoWhisper-large",  "language": "vietnamese", "category": "vietnamese-specific", "size": "1550M"},
    {"id": "vinai/PhoWhisper-medium", "language": "vietnamese", "category": "vietnamese-specific", "size": "769M"},
    {"id": "vinai/PhoWhisper-small",  "language": "vietnamese", "category": "vietnamese-specific", "size": "244M"},
    {"id": "vinai/PhoWhisper-base",   "language": "vietnamese", "category": "vietnamese-specific", "size": "74M"},
    {"id": "vinai/PhoWhisper-tiny",   "language": "vietnamese", "category": "vietnamese-specific", "size": "39M"},
    # General multilingual (OpenAI Whisper)
    {"id": "openai/whisper-large-v3",       "language": "vietnamese", "category": "multilingual", "size": "1550M"},
    {"id": "openai/whisper-large-v3-turbo", "language": "vietnamese", "category": "multilingual", "size": "809M"},
    {"id": "openai/whisper-medium",          "language": "vietnamese", "category": "multilingual", "size": "769M"},
    {"id": "openai/whisper-small",           "language": "vietnamese", "category": "multilingual", "size": "244M"},
    {"id": "openai/whisper-base",            "language": "vietnamese", "category": "multilingual", "size": "74M"},
    {"id": "openai/whisper-tiny",            "language": "vietnamese", "category": "multilingual", "size": "39M"},
]

# === DATASETS ===
# Add more datasets here as needed
DATASETS = [
    {"id": "thanhnew2001/VietSuperSpeech", "split": "validation", "name": "VietSuperSpeech"},
    # {"id": "your-dataset-id", "split": "test", "name": "YourDataset"},
]

# === SETTINGS ===
BATCH_SIZE = 8
MAX_SAMPLES = None  # None = all samples. Set to e.g. 100 for quick testing

# === HUGGINGFACE HUB (optional) ===
PUSH_TO_HF = False  # Set True to auto-push results
HF_REPO_ID = ""     # e.g. "your-username/asr-benchmark-results"

# GPU pricing (USD/hour) for cost estimation
GPU_PRICING = {"T4": 0.35, "P100": 0.46, "free_tier": 0.00}

print(f"Models: {len(MODELS)}")
print(f"Datasets: {len(DATASETS)}")
print(f"Max samples: {MAX_SAMPLES or 'all'}")

## 4. Benchmark Engine

In [ ]:
import gc
import io
import json
import time
from datetime import datetime, timezone

import librosa
import numpy as np
import requests
from datasets import load_dataset
from huggingface_hub import hf_hub_url
from jiwer import wer, cer
from tqdm.auto import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

TARGET_SR = 16000

def normalize_text(text):
    return " ".join(text.strip().lower().split())

def download_audio(audio_path, dataset_id):
    url = hf_hub_url(repo_id=dataset_id, filename=audio_path, repo_type="dataset")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    arr, _ = librosa.load(io.BytesIO(resp.content), sr=TARGET_SR, mono=True)
    return arr

def load_asr_pipeline(model_id, language="vietnamese"):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True)
    model.to(device)
    processor = AutoProcessor.from_pretrained(model_id)
    pipe = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
                    feature_extractor=processor.feature_extractor, torch_dtype=dtype, device=device)
    return pipe, model

def unload_model(pipe, model):
    del pipe, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def benchmark_model(model_cfg, dataset, dataset_id, batch_size, max_samples):
    model_id = model_cfg["id"]
    language = model_cfg.get("language", "vietnamese")
    print(f"\n{'='*60}\nBenchmarking: {model_id}\n{'='*60}")
    
    load_start = time.time()
    pipe, model = load_asr_pipeline(model_id, language)
    model_load_time = time.time() - load_start
    
    num = min(len(dataset), max_samples) if max_samples else len(dataset)
    refs, preds, samples = [], [], []
    total_audio, total_infer = 0.0, 0.0
    
    for i in tqdm(range(0, num, batch_size), desc=model_id.split('/')[-1]):
        batch = dataset.select(list(range(i, min(i + batch_size, num))))
        arrs, texts, durs = [], [], []
        for s in batch:
            try:
                arrs.append(download_audio(s["audio"], dataset_id))
                texts.append(s["text"])
                durs.append(s.get("duration", 0.0))
            except Exception as e:
                print(f"Skip: {e}")
        if not arrs: continue
        
        t0 = time.time()
        results = pipe(arrs, batch_size=len(arrs), generate_kwargs={"language": language, "task": "transcribe"})
        infer_t = time.time() - t0
        total_infer += infer_t
        
        for j, (r, ref, dur) in enumerate(zip(results, texts, durs)):
            rn, pn = normalize_text(ref), normalize_text(r["text"])
            sw = wer(rn, pn) if rn else 0.0
            sc = cer(rn, pn) if rn else 0.0
            total_audio += dur
            refs.append(rn); preds.append(pn)
            samples.append({"index": i+j, "reference": rn, "prediction": pn,
                           "wer": round(sw, 4), "cer": round(sc, 4), "duration_s": round(dur, 2)})
    
    ow = wer(refs, preds) if refs else 1.0
    oc = cer(refs, preds) if refs else 1.0
    rtf = total_infer / total_audio if total_audio > 0 else None
    costs = {gpu: round((total_infer/3600)*price, 6) for gpu, price in GPU_PRICING.items()}
    
    print(f"  WER: {ow:.4f} | CER: {oc:.4f} | RTF: {rtf:.4f if rtf else 'N/A'} | Cost(T4): ${costs.get('T4', 0):.6f}")
    unload_model(pipe, model)
    
    return {
        "model_id": model_id, "category": model_cfg.get("category", ""),
        "size_params": model_cfg.get("size", ""), "dataset_id": dataset_id,
        "metrics": {"wer": round(ow,4), "cer": round(oc,4), "num_samples": len(refs),
                    "total_audio_duration_s": round(total_audio,2), "total_inference_time_s": round(total_infer,2),
                    "model_load_time_s": round(model_load_time,2),
                    "real_time_factor": round(rtf,4) if rtf else None,
                    "avg_time_per_sample_s": round(total_infer/len(refs),4) if refs else None},
        "cost_estimate_usd": costs, "per_sample_results": samples,
    }

print("Benchmark engine ready.")

## 5. Run Full Benchmark

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
device = "cuda" if torch.cuda.is_available() else "cpu"
all_results = []
leaderboard = []

for ds_cfg in DATASETS:
    dataset_id = ds_cfg["id"]
    split = ds_cfg.get("split", "validation")
    print(f"\nLoading dataset: {dataset_id} ({split})...")
    dataset = load_dataset(dataset_id, split=split)
    print(f"Loaded: {len(dataset)} samples")
    
    for model_cfg in MODELS:
        try:
            result = benchmark_model(model_cfg, dataset, dataset_id, BATCH_SIZE, MAX_SAMPLES)
            all_results.append(result)
            leaderboard.append({
                "model_id": result["model_id"], "category": result["category"],
                "size_params": result["size_params"], "dataset_id": result["dataset_id"],
                "wer": result["metrics"]["wer"], "cer": result["metrics"]["cer"],
                "rtf": result["metrics"]["real_time_factor"],
                "inference_time_s": result["metrics"]["total_inference_time_s"],
                "avg_time_per_sample_s": result["metrics"]["avg_time_per_sample_s"],
                "cost_T4_usd": result["cost_estimate_usd"].get("T4", 0),
                "cost_free_tier": result["cost_estimate_usd"].get("free_tier", 0),
            })
        except Exception as e:
            print(f"FAILED: {model_cfg['id']} - {e}")
            all_results.append({"model_id": model_cfg["id"], "dataset_id": dataset_id, "error": str(e)})

leaderboard.sort(key=lambda x: x.get("wer", 999))
print("\nBenchmark complete!")

## 6. Leaderboard

In [ ]:
import pandas as pd

lb_df = pd.DataFrame(leaderboard)
lb_df.index = range(1, len(lb_df) + 1)
lb_df.index.name = "Rank"

print("\n" + "="*80)
print("LEADERBOARD (sorted by WER)")
print("="*80)
display(lb_df[["model_id", "category", "size_params", "wer", "cer", "rtf", "inference_time_s", "cost_T4_usd"]])

## 7. Save Results

In [ ]:
full_output = {
    "metadata": {
        "timestamp": timestamp, "device": device,
        "torch_version": torch.__version__,
        "num_models": len(MODELS), "num_datasets": len(DATASETS),
        "max_samples": MAX_SAMPLES, "batch_size": BATCH_SIZE,
    },
    "leaderboard": leaderboard,
    "detailed_results": all_results,
}

output_file = f"/kaggle/working/benchmark_{timestamp}.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_output, f, ensure_ascii=False, indent=2)
print(f"Results saved to {output_file}")

# Also save leaderboard CSV for easy viewing
csv_file = f"/kaggle/working/leaderboard_{timestamp}.csv"
lb_df.to_csv(csv_file)
print(f"Leaderboard CSV: {csv_file}")

## 8. Visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# WER comparison
ax = axes[0, 0]
colors = ['#2ecc71' if cat == 'vietnamese-specific' else '#3498db' for cat in lb_df['category']]
ax.barh(lb_df['model_id'], lb_df['wer'], color=colors)
ax.set_xlabel('WER (lower = better)')
ax.set_title('Word Error Rate by Model')
ax.invert_yaxis()

# CER comparison
ax = axes[0, 1]
ax.barh(lb_df['model_id'], lb_df['cer'], color=colors)
ax.set_xlabel('CER (lower = better)')
ax.set_title('Character Error Rate by Model')
ax.invert_yaxis()

# RTF comparison
ax = axes[1, 0]
rtf_df = lb_df.dropna(subset=['rtf'])
rtf_colors = ['#2ecc71' if cat == 'vietnamese-specific' else '#3498db' for cat in rtf_df['category']]
ax.barh(rtf_df['model_id'], rtf_df['rtf'], color=rtf_colors)
ax.set_xlabel('Real-Time Factor (lower = faster)')
ax.set_title('Inference Speed')
ax.axvline(x=1.0, color='red', linestyle='--', label='Real-time')
ax.legend()
ax.invert_yaxis()

# Cost vs Quality
ax = axes[1, 1]
for cat in lb_df['category'].unique():
    cat_df = lb_df[lb_df['category'] == cat]
    color = '#2ecc71' if cat == 'vietnamese-specific' else '#3498db'
    ax.scatter(cat_df['cost_T4_usd'], cat_df['wer'], label=cat, color=color, s=100)
    for _, row in cat_df.iterrows():
        ax.annotate(row['model_id'].split('/')[-1], (row['cost_T4_usd'], row['wer']),
                   fontsize=7, ha='left', va='bottom')
ax.set_xlabel('Cost (USD, T4 GPU)')
ax.set_ylabel('WER (lower = better)')
ax.set_title('Cost vs Quality Trade-off')
ax.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/benchmark_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved.")

## 9. Push to HuggingFace (Optional)

In [ ]:
if PUSH_TO_HF and HF_REPO_ID:
    from huggingface_hub import HfApi, create_repo, login
    from datasets import Dataset, DatasetDict
    import os
    
    # Try Kaggle secrets first, then env var
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        login(token=token)
    except:
        token = os.environ.get("HF_TOKEN")
        if token:
            login(token=token)
        else:
            print("No HF_TOKEN found. Add it in Kaggle Settings > Secrets.")
    
    create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
    
    lb_ds = Dataset.from_dict({k: [e.get(k) for e in leaderboard] for k in [
        "model_id", "category", "size_params", "dataset_id",
        "wer", "cer", "rtf", "inference_time_s",
        "avg_time_per_sample_s", "cost_T4_usd", "cost_free_tier"]})
    
    all_samples = []
    for d in all_results:
        if "error" in d: continue
        for s in d.get("per_sample_results", []):
            all_samples.append({"model_id": d["model_id"], "dataset_id": d["dataset_id"], **s})
    
    samples_ds = Dataset.from_dict({k: [s.get(k) for s in all_samples] for k in [
        "model_id", "dataset_id", "index", "reference", "prediction", "wer", "cer", "duration_s"]})
    
    DatasetDict({"leaderboard": lb_ds, "per_sample": samples_ds}).push_to_hub(HF_REPO_ID)
    
    api = HfApi()
    api.upload_file(path_or_fileobj=output_file, path_in_repo="latest_benchmark.json",
                    repo_id=HF_REPO_ID, repo_type="dataset")
    api.upload_file(path_or_fileobj="/kaggle/working/benchmark_charts.png", path_in_repo="benchmark_charts.png",
                    repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"Pushed to https://huggingface.co/datasets/{HF_REPO_ID}")
else:
    print("Set PUSH_TO_HF=True and HF_REPO_ID to push results.")
    print(f"Download results from: {output_file}")

## 10. Summary & Recommendations

In [ ]:
print("\n" + "="*60)
print("BENCHMARK SUMMARY & RECOMMENDATIONS")
print("="*60)

if leaderboard:
    best = leaderboard[0]
    print(f"\nBest overall model: {best['model_id']}")
    print(f"  WER: {best['wer']:.4f} ({best['wer']*100:.2f}%)")
    print(f"  CER: {best['cer']:.4f} ({best['cer']*100:.2f}%)")
    
    # Best by category
    vn_models = [m for m in leaderboard if m.get('category') == 'vietnamese-specific']
    ml_models = [m for m in leaderboard if m.get('category') == 'multilingual']
    
    if vn_models:
        best_vn = vn_models[0]
        print(f"\nBest Vietnamese-specific: {best_vn['model_id']} (WER: {best_vn['wer']:.4f})")
    if ml_models:
        best_ml = ml_models[0]
        print(f"Best Multilingual: {best_ml['model_id']} (WER: {best_ml['wer']:.4f})")
    
    # Best cost-effective (lowest WER among small models)
    small_models = [m for m in leaderboard if 'tiny' in m['model_id'] or 'base' in m['model_id'] or 'small' in m['model_id']]
    if small_models:
        best_small = min(small_models, key=lambda x: x.get('wer', 999))
        print(f"Best cost-effective (small): {best_small['model_id']} (WER: {best_small['wer']:.4f})")
    
    # Fastest
    fastest = min([m for m in leaderboard if m.get('rtf')], key=lambda x: x['rtf'], default=None)
    if fastest:
        print(f"Fastest (lowest RTF): {fastest['model_id']} (RTF: {fastest['rtf']:.4f})")

print("\n" + "="*60)
print("To schedule auto-runs: File > Schedule > Set frequency")
print("To add more datasets: Edit DATASETS list in cell 3")
print("="*60)